### Feature Engineering

In [1]:
# Step 1: Feature Engineering

import pandas as pd
from pathlib import Path

# Define input and output paths
input_path = Path("../data/processed/transactions_cleaned.csv")
output_path = Path("../data/processed/transactions_featured.csv")

# Check input file
print("Input dataset exists:", input_path.exists())

# Initialize counters
total_rows = 0
first_chunk = True

# Process dataset in chunks
for chunk in pd.read_csv(input_path, chunksize=500_000):

    total_rows += len(chunk)

    # Feature 1: Change in sender balance
    chunk["origin_balance_change"] = (
        chunk["oldbalanceOrg"] - chunk["newbalanceOrig"]
    )

    # Feature 2: Change in receiver balance
    chunk["destination_balance_change"] = (
        chunk["newbalanceDest"] - chunk["oldbalanceDest"]
    )

    # Feature 3: Sender balance error
    chunk["errorBalanceOrig"] = (
        chunk["oldbalanceOrg"]
        - chunk["amount"]
        - chunk["newbalanceOrig"]
    )

    # Feature 4: Receiver balance error
    chunk["errorBalanceDest"] = (
        chunk["oldbalanceDest"]
        + chunk["amount"]
        - chunk["newbalanceDest"]
    )

    # Feature 5: Hour of the transaction (0 to 23)
    chunk["hour"] = (chunk["step"] - 1) % 24

    # Feature 6: Day of the simulation (starting from day 1)
    chunk["day"] = ((chunk["step"] - 1) // 24) + 1

    # Remove customer identifiers
    chunk = chunk.drop(columns=["nameOrig", "nameDest"])

    # Save the engineered data
    chunk.to_csv(
        output_path,
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False

print("\nFEATURE ENGINEERING COMPLETED")
print("Total transactions processed:", total_rows)
print("Output file:", output_path)

Input dataset exists: True

FEATURE ENGINEERING COMPLETED
Total transactions processed: 6362620
Output file: ..\data\processed\transactions_featured.csv


In [2]:
# Step 2: Verify the engineered dataset

import pandas as pd
from pathlib import Path

featured_path = Path("../data/processed/transactions_featured.csv")

print("Featured dataset exists:", featured_path.exists())

# Read only the first 5 rows
df = pd.read_csv(featured_path, nrows=5)

print("\nSample shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nNew engineered features:")
new_features = [
    "origin_balance_change",
    "destination_balance_change",
    "errorBalanceOrig",
    "errorBalanceDest",
    "hour",
    "day"
]

print(df[new_features])

print("\nCustomer identifiers removed:",
      "nameOrig" not in df.columns and "nameDest" not in df.columns)

Featured dataset exists: True

Sample shape: (5, 15)

Columns:
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'origin_balance_change', 'destination_balance_change', 'errorBalanceOrig', 'errorBalanceDest', 'hour', 'day']

New engineered features:
   origin_balance_change  destination_balance_change  errorBalanceOrig  \
0                9839.64                         0.0               0.0   
1                1864.28                         0.0               0.0   
2                 181.00                         0.0               0.0   
3                 181.00                    -21182.0               0.0   
4               11668.14                         0.0               0.0   

   errorBalanceDest  hour  day  
0           9839.64     0    1  
1           1864.28     0    1  
2            181.00     0    1  
3          21363.00     0    1  
4          11668.14     0    1  

Customer identifiers removed: 

In [1]:
import sys
print(sys.executable)

e:\ml project\Fraud-Detection\fraud-detection-mlops\.venv\Scripts\python.exe


In [2]:
# Step 3: Train / Validation / Test Split

import pandas as pd
import numpy as np
from pathlib import Path

# Input and output paths
input_path = Path("../data/processed/transactions_featured.csv")
output_dir = Path("../data/processed")

print("Input dataset exists:", input_path.exists())

if not input_path.exists():
    raise FileNotFoundError(f"Dataset not found: {input_path}")

# Random generator for reproducible splitting
rng = np.random.default_rng(42)

# Split ratios
train_ratio = 0.70
validation_ratio = 0.15

split_files = {
    "train": output_dir / "train.csv",
    "validation": output_dir / "validation.csv",
    "test": output_dir / "test.csv"
}

row_counts = {name: 0 for name in split_files}
fraud_counts = {name: 0 for name in split_files}
first_chunk = True

# Process the dataset in chunks
for chunk in pd.read_csv(input_path, chunksize=500_000):

    random_values = rng.random(len(chunk))

    train_mask = random_values < train_ratio

    validation_mask = (
        (random_values >= train_ratio) &
        (random_values < train_ratio + validation_ratio)
    )

    test_mask = random_values >= train_ratio + validation_ratio

    masks = {
        "train": train_mask,
        "validation": validation_mask,
        "test": test_mask
    }

    for name, mask in masks.items():
        split_df = chunk.loc[mask]

        split_df.to_csv(
            split_files[name],
            index=False,
            mode="w" if first_chunk else "a",
            header=first_chunk
        )

        row_counts[name] += len(split_df)
        fraud_counts[name] += int(split_df["isFraud"].sum())

    first_chunk = False

# Display results
print("\nDATA SPLITTING COMPLETED")

for name in split_files:
    fraud_rate = fraud_counts[name] / row_counts[name] * 100

    print(f"\n{name.upper()}")
    print("Transactions:", row_counts[name])
    print("Fraud transactions:", fraud_counts[name])
    print(f"Fraud percentage: {fraud_rate:.4f}%")
    print("File:", split_files[name])

Input dataset exists: True

DATA SPLITTING COMPLETED

TRAIN
Transactions: 4453776
Fraud transactions: 5727
Fraud percentage: 0.1286%
File: ..\data\processed\train.csv

VALIDATION
Transactions: 954744
Fraud transactions: 1181
Fraud percentage: 0.1237%
File: ..\data\processed\validation.csv

TEST
Transactions: 954100
Fraud transactions: 1305
Fraud percentage: 0.1368%
File: ..\data\processed\test.csv
